In [25]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")

import gdsfactory as gf
# TODO
# 1. Change one side of the cell_temp to be comparation with the width
# 2. Different with the width and same width with gap

cell_temp = gf.Component()

In [26]:
# cell
L = [500, 1000, 2000]
g = 48
column_list = [2, 2, 1]
w = 2
all_lines =[None] * len(L)
for i in range(len(L)):
    mesh = [None] * column_list[i]
    for j in range(column_list[i]):
        mesh[j] = gf.Component()    
    all_lines[i] = mesh
block = gf.Component()
origin = [[0,0], [0,0], [2500 - L[2]/2,0]]
x_pace = [5000, 5000, 0]
y_pace = [0, 4000+w, 10000-L[2]]
for i in range(len(L)):
    for j in range(column_list[i]):
        x_move = origin[i][0]+(x_pace[i]-L[i])*j
        y_move = origin[i][1]+y_pace[i]
        # single grid
        row = (L[i]) // (g + w)
        sg = gf.components.rectangle(size=(g, g), layer=(8, 0))
        for m in range(row):
            for n in range(row):
                sf_ref = all_lines[i][j] << sg
                sf_ref.move((m*(g+w), +n*(g+w)))
        # frame of the grid
        frame = gf.components.rectangle(size=(L[i]+w, L[i]-w), layer=(9, 0))
        (all_lines[i][j] << frame).movex(-w)
        # reflection area
        fra_width = L[i] + w
        x_mid = - w + fra_width / 2
        y_mid = (L[i]-w) / 2
        cry_size = 50
        # cry_height = height+0.1 if j != 0 else 2*height+w+0.1
        crystal = gf.components.rectangle(size=(cry_size, cry_size), layer=(10, 0))
        crystal_ref = all_lines[i][j] << crystal
        crystal_ref.move((x_mid-cry_size/2, y_mid-cry_size/2))
        # deposition area
        gold_size = cry_size - 6
        gold = gf.components.rectangle(size=(gold_size, gold_size), layer=(11, 0))
        gold_ref = all_lines[i][j] << gold
        gold_ref.move((x_mid-gold_size/2, y_mid-gold_size/2))

        # frontside etching area
        front_etch_gap = 1000  # double sides gap total
        frontframe_etch = gf.components.rectangle(size=(fra_width+front_etch_gap, L[i]-w), layer=(12, 0))
        (all_lines[i][j] << frontframe_etch).movex((-w-front_etch_gap/2))
        # backside etching area
        back_etch_gap = 743.44  # double sides gap total
        backframe_etch = gf.components.rectangle(size=(fra_width+front_etch_gap+back_etch_gap, L[i]-w+back_etch_gap), layer=(3, 0))
        (all_lines[i][j] << backframe_etch).move((-w - back_etch_gap/2-front_etch_gap/2, -back_etch_gap/2))
        
        # length mark
        T = gf.components.text(f"L={L[i]} gap={g}", size=50, layer=(1, 0))
        T_ref = all_lines[i][j] << T
        T_ref.move((L[i]/2-300, -200))
        (block << all_lines[i][j]).move((x_move, y_move))

n_block = 2
# block.show()

In [27]:
# Note for block/fblock cell
"""
layer 8: grid
layer 9: frame (outside sides of the mesh)
layer 10: deposition area on the mesh
layer 11: gold deposition area
layer 12: frontside etching area (in other design layer 3)
layer 3: backside etching area (in individual cell named backside in other designs)
"""

'\nlayer 8: grid\nlayer 9: frame (outside sides of the mesh)\nlayer 10: deposition area on the mesh\nlayer 11: gold deposition area\nlayer 12: frontside etching area (in other design layer 3)\nlayer 3: backside etching area (in individual cell named backside in other designs)\n'

In [28]:
# structure for each die (2 copy of structures)
fblock = gf.Component()
for i in range(n_block):
    if i == 0:
        block_ref = fblock << block
    elif i == 1:
        block_ref = fblock << block
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block
        block_ref.move((0, 10000))

In [ ]:
# order
order = gf.Component()
text_array = []
for i in range(4):
    text_array.append(gf.Component())

for i in range(n_block):
    # DML: double-clamped mesh large
    T = gf.components.text(f"US {i+1}", size=20, layer=(1, 0))
    for j in range(4):
        order_ref = text_array[i] << T
        if j == 0:
            order_ref.move((-185, -100))
        elif j == 1:
            order_ref.move((-85, 5300))
        elif j == 2:
            order_ref.move((5020, 0))
        else:
            order_ref.move((5020, 5300))
    text_array_ref = order << text_array[i]
    if i == 0:
        pass
    elif i == 1:
        text_array_ref.move((10000, 0))
    elif i == 2:
        text_array_ref.move((0, 10000))
    else:
        text_array_ref.move((10000, 10000))





In [30]:
# boolean operation
# do frontside etching - outside frame of mesh
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(12, 0), layer2=(9, 0), layer=(1, 0))

cell_temp << outside

# add holes(with gold position) to cell_temp
holes = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(8, 0), layer2=(10, 0), layer=(1, 0))
cell_temp << holes

# add gold
pc = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(11, 0), layer2=(30, 0), layer=(2, 0))
cell_temp << pc

# backside etching
back = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(30, 0), layer=(3, 0))
cell_temp << back
# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(10, 0), layer=(1, 0))
cell_temp << marker
cell_temp << order
# frame
frame1 = gf.components.rectangle(size=(15000, 10830), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 15830), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_ultrasound = gf.Component()
cell_ref = cell_ultrasound << cell_temp
cell_ref.move((2500, -7500))
cell_ultrasound.show()
# cell_ultrasound.write_gds("mesh.gds")
# cell_ultrasound.plot()